In [2]:
import polars as pl

In [3]:
inicial = (
    pl.scan_parquet(r"C:\Users\Usuario\Downloads\covid_sp.parquet")
    .select([
        "paciente_idade",
        "paciente_enumSexoBiologico",
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf",
        "vacina_dataAplicacao",
        "vacina_fabricante_nome",
        "vacina_categoria_nome"
    ])
    .with_columns([
        pl.col("paciente_idade").cast(pl.Int64, strict=False),
        pl.col("vacina_dataAplicacao").str.to_date("%Y-%m-%d", strict=False),
        pl.lit("SP").alias("estado"),
        pl.lit("Sudeste").alias("regiao")
    ])
    .filter(
        pl.col("paciente_idade").is_not_null() &
        pl.col("paciente_endereco_nmMunicipio").is_not_null() &
        pl.col("paciente_endereco_uf").is_not_null()
    )
)

display(inicial.head(5).collect())

paciente_idade,paciente_enumSexoBiologico,paciente_endereco_nmMunicipio,paciente_endereco_uf,vacina_dataAplicacao,vacina_fabricante_nome,vacina_categoria_nome,estado,regiao
i64,str,str,str,date,str,str,str,str
21,"""M""","""INDAIATUBA""","""SP""",2021-09-16,"""SINOVAC/BUTANTAN""","""Faixa Etária""","""SP""","""Sudeste"""
3,"""F""","""SAO JOSE DO RIO PRETO""","""SP""",2022-11-22,"""SINOVAC/BUTANTAN""","""""","""SP""","""Sudeste"""
45,"""F""","""MOGI DAS CRUZES""","""SP""",2022-07-20,"""ASTRAZENECA/FIOCRUZ""","""Faixa Etária""","""SP""","""Sudeste"""
50,"""M""","""CAIEIRAS""","""SP""",2022-07-14,"""ASTRAZENECA/FIOCRUZ""","""Faixa Etária""","""SP""","""Sudeste"""
24,"""M""","""RIBEIRAO PRETO""","""SP""",2021-09-02,"""SINOVAC/BUTANTAN""","""Faixa Etária""","""SP""","""Sudeste"""


In [4]:
df_municipios = (
    inicial
    .group_by(
        "paciente_endereco_nmMunicipio",
        "paciente_endereco_uf",
        "estado",
        "regiao"
    )
    .agg([
        pl.len().alias("total_vacinacoes"),

        pl.col("paciente_idade")
        .mean()
        .round(2)
        .alias("media_idade")
    ])
)

In [5]:
df_agregado = df_municipios.collect()

In [6]:
# 3. Calcular quartis

q1 = (
    df_agregado
    .select(
        pl.col("total_vacinacoes")
        .quantile(0.25)
    )
    .item()
)

q3 = (
    df_agregado
    .select(
        pl.col("total_vacinacoes")
        .quantile(0.75)
    )
    .item()
)

print(f"Municípios abaixo de {q1:.0f} vacinações são 'Baixa Vacinação'")
print(f"Municípios acima de {q3:.0f} vacinações são 'Alta Vacinação'")

Municípios abaixo de 27 vacinações são 'Baixa Vacinação'
Municípios acima de 499 vacinações são 'Alta Vacinação'


In [7]:
# 4. Criar classificação

df_export = (
    df_agregado
    .with_columns(
        pl.when(
            pl.col("total_vacinacoes") > q3
        )
        .then(pl.lit("Alta Vacinação"))
        
        .when(
            pl.col("total_vacinacoes") < q1
        )
        .then(pl.lit("Baixa Vacinação"))
        
        .otherwise(pl.lit("Média Vacinação"))
        
        .alias("Etiqueta_Classificacao")
    )
)

In [8]:
# 5. Exportar CSV

df_export.write_csv("vacinas_sp_powerbi.csv")

print("Arquivo 'vacinas_sp_powerbi.csv' gerado com sucesso!")

Arquivo 'vacinas_sp_powerbi.csv' gerado com sucesso!
